# Inspect Overture buildings GeoParquet files

This notebook targets the `theme=buildings/type=building` release folder, reports schema details, and previews a few rows from the GeoParquet files using DuckDB. All files share the same structure, so a single sample per directory is sufficient.

In [ ]:
from pathlib import Path
import sys

def find_repo_root(start: Path) -> Path:
    for parent in (start, *start.parents):
        if (parent / '.git').exists():
            return parent
    raise RuntimeError(f'Could not find repository root from {start}')

REPO_ROOT = find_repo_root(Path.cwd().resolve())
DATA_DIR = REPO_ROOT / 'data'
RESULTS_DIR = DATA_DIR / 'results'
GIS_DIR = REPO_ROOT / 'gis_data'
MODULE_ROOT = REPO_ROOT / 'code' / 'overture_analysis'

if str(MODULE_ROOT) not in sys.path:
    sys.path.append(str(MODULE_ROOT))

RESULTS_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
from collections import defaultdict

import duckdb
from IPython.display import display

BUILDINGS_RELEASE = '2025-10-22.0'
BASE_PATH = GIS_DIR / 'overturemaps-us-west-2' / 'release' / BUILDINGS_RELEASE / 'theme=buildings' / 'type=building'
print(f'Base directory: {BASE_PATH}')

parquet_files = sorted(BASE_PATH.rglob('*.parquet'))
if not parquet_files:
    raise FileNotFoundError('No GeoParquet files found in theme=buildings/type=building.')

files_by_dir = defaultdict(list)
for file_path in parquet_files:
    files_by_dir[file_path.parent].append(file_path)

print(f'Found {len(parquet_files)} parquet files across {len(files_by_dir)} directories.')
for directory, files in sorted(files_by_dir.items()):
    rel_dir = directory.relative_to(BASE_PATH)
    print(f'{rel_dir}: {len(files)} file(s)')


In [ ]:
import json
from pyarrow import parquet as pq

sample_file_for_metadata = parquet_files[0]
print(f"Inspecting metadata for: {sample_file_for_metadata.name}")
pq_file = pq.ParquetFile(sample_file_for_metadata)
key_value_metadata = pq_file.metadata.metadata or {}

def _decode_if_bytes(value):
    return value.decode('utf-8', 'replace') if isinstance(value, (bytes, bytearray)) else value

decoded_metadata = {
    _decode_if_bytes(k): _decode_if_bytes(v)
    for k, v in key_value_metadata.items()
}
print(json.dumps(decoded_metadata, indent=2))

geo_metadata = decoded_metadata.get('geo')
if geo_metadata:
    try:
        geo_json = json.loads(geo_metadata)
        primary_column = geo_json.get('primary_column')
        columns = geo_json.get('columns', {})
        geometry_info = columns.get(primary_column, {}) if columns else {}
        bbox = geometry_info.get('bbox')
        if bbox:
            print('Geometry bounding box:', bbox)
    except json.JSONDecodeError:
        print('Unable to parse geo metadata as JSON.')


In [ ]:
con = duckdb.connect(database=':memory:')

for directory, files in sorted(files_by_dir.items()):
    sample_file = files[0]
    rel_dir = directory.relative_to(BASE_PATH)
    rel_label = '.' if str(rel_dir) == '.' else str(rel_dir)
    print(f"
=== {rel_label} ===")
    print(f"Sample file: {sample_file.name}")
    schema_df = con.execute(
        "DESCRIBE SELECT * FROM read_parquet(?)", [str(sample_file)]
    ).fetchdf()
    display(schema_df)
    sample_rows_df = con.execute(
        "SELECT * FROM read_parquet(?) LIMIT 5", [str(sample_file)]
    ).fetchdf()
    display(sample_rows_df)
    print('Sources column sample:')
    for idx, value in sample_rows_df['sources'].items():
        print(f"Row {idx}: {value}")
